In [ ]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
import os
from pathlib import Path
import joblib
from sklearn.pipeline import Pipeline
import optuna
from optuna.samplers import TPESampler

In [ ]:
put = Path('/content/drive/MyDrive/classic_data.parquet') #вставить свой путь к classic_data
df = pd.read_parquet(put)

# Деление выборки и векторизация текстов

In [ ]:
X = df['text_lemma']
y = df['topic']
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
vectorizer = TfidfVectorizer(max_features=20000, min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

##Подбор параметров для модели

In [ ]:
from optuna.samplers import TPESampler

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    pipeline = Pipeline([
        ('clf', LogisticRegression(
            C=trial.suggest_float('C', 0.05, 20.0, log=True),
            class_weight='balanced',
            max_iter=1000,
            solver='lbfgs',
        )),
    ])
    scores = cross_val_score(pipeline, X_train_tfidf, y_train, cv=cv, scoring='f1_macro', n_jobs=-1,error_score='raise')
    return scores.mean()

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42, multivariate=True),
)
study.optimize(objective, n_trials=15, show_progress_bar=True,timeout=3600)
best_params = study.best_params
print(f'Лучший CV macro-F1: {study.best_value:.4f}')
print('Лучшие гиперпараметры:', best_params)

[I 2026-10-07 08:07:20,531] A new study created in memory with name: no-name-36008025-4f6a-4cc7-81bf-1b7074303dfd


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-10-07 08:08:28,339] Trial 0 finished with value: 0.7008880789069313 and parameters: {'C': 0.47156966780898374}. Best is trial 0 with value: 0.7008880789069313.
[I 2026-10-07 08:10:48,515] Trial 1 finished with value: 0.7427933393123981 and parameters: {'C': 14.886262201211787}. Best is trial 1 with value: 0.7427933393123981.
[I 2026-10-07 08:12:40,664] Trial 2 finished with value: 0.7394060253324154 and parameters: {'C': 4.014783718209775}. Best is trial 1 with value: 0.7427933393123981.
[I 2026-10-07 08:14:06,944] Trial 3 finished with value: 0.7289342118449144 and parameters: {'C': 1.805989805778526}. Best is trial 1 with value: 0.7427933393123981.
[I 2026-10-07 08:14:40,171] Trial 4 finished with value: 0.6606803871048735 and parameters: {'C': 0.12733267608983528}. Best is trial 1 with value: 0.7427933393123981.
[I 2026-10-07 08:15:13,649] Trial 5 finished with value: 0.6607048286711993 and parameters: {'C': 0.12731427596811182}. Best is trial 1 with value: 0.742793339312398

# Модель и обучение

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced',C=9.2,solver='lbfgs')
model1 = LogisticRegression(max_iter=1000, class_weight='balanced',C=9.2,solver='lbfgs')
model.fit(X_train_tfidf, y_train)

LogisticRegression(C=9.2, class_weight='balanced', max_iter=1000)

In [ ]:
y_pred = model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred,zero_division=0))

                   precision    recall  f1-score   support

   69-я параллель       0.66      0.62      0.64        34
           Бизнес       0.48      0.65      0.55       200
      Бывший СССР       0.82      0.87      0.85      1446
              Дом       0.79      0.83      0.81       589
         Из жизни       0.58      0.70      0.63       748
   Интернет и СМИ       0.76      0.75      0.75      1210
             Крым       0.32      0.44      0.37        18
         Культура       0.88      0.89      0.88      1457
              Мир       0.84      0.80      0.82      3700
  Наука и техника       0.83      0.83      0.83      1439
      Путешествия       0.68      0.76      0.72       174
           Россия       0.84      0.76      0.80      4345
Силовые структуры       0.51      0.68      0.58       531
            Спорт       0.95      0.97      0.96      1745
         Ценности       0.86      0.90      0.88       210
        Экономика       0.85      0.85      0.85      2

# Сохранение модели и векторизации

In [ ]:
Put = Path('/content/drive/MyDrive') # написать путь сохранения
Put.mkdir(exist_ok=True)
joblib.dump(vectorizer,Put/'tfidf_vector.joblib')
joblib.dump(model,Put/'classic_model.joblib')

['/content/drive/MyDrive/classic_model.joblib']